In [1]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

/Users/rahuljauhari/Rahul Jauhari/Personal Projects/GenAI - Learning/Langchain/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/rahuljauhari/Rahul Jauhari/Personal Projects/GenAI - Learning/Langchain/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


# Parallel Chain — Running Multiple Chains Simultaneously

## What is a Parallel Chain?

A **parallel chain** runs multiple chains on the **same input at the same time**, and collects all their outputs into a dictionary.

Instead of:
```
input → chain_A → result_A
input → chain_B → result_B   (you run these one after another — slow)
```

You get:
```
              ┌→ chain_A (generate notes) →┐
input ─────────┤                            ├→ {"notes": ..., "questions": ...}
              └→ chain_B (generate quiz)  →┘
(both run simultaneously — faster!)
```

## When to use it

Use `RunnableParallel` when:
- You need **multiple perspectives** on the same data (e.g. notes + quiz + summary)
- The tasks are **independent** — neither needs the result of the other
- **Speed matters** — parallel execution can be significantly faster than sequential

## How it works

```python
from langchain_core.runnables import RunnableParallel

parallel = RunnableParallel({
    "notes":     notes_prompt     | model | StrOutputParser(),
    "questions": questions_prompt | model | StrOutputParser(),
})

result = parallel.invoke({"data": "some text"})
print(result["notes"])      # notes about the text
print(result["questions"])  # questions about the text
```

The keys in the dictionary become the keys in the output.

## What this notebook builds

This notebook takes a long text and runs two chains in parallel:
1. **Notes chain** — generates structured notes
2. **Questions chain** — generates study questions

Then a third **aggregator chain** merges both outputs into a single document.

## What you'll learn

- How to use `RunnableParallel` to run multiple chains at once
- How to combine parallel outputs with a follow-up sequential chain
- How to pass large text inputs through a chain

## Prerequisites

- Ollama running with a model pulled (uses `qwen3-coder:30b` — change to any model you have)
- Virtual environment activated

In [2]:
model = ChatOllama(model="qwen3-coder:30b")

In [3]:
promptGenerateNotes = PromptTemplate(
    input_variables=["data"],
    template="Generate notes for the text: {data}"
)

In [4]:
promptGenerateQuestions = PromptTemplate(
    input_variables=["data"],
    template="Generate questions for the text: {data}"
)

In [5]:
parser = StrOutputParser()

In [6]:
promptAggregator = PromptTemplate(
    input_variables=["notes", "questions"],
    template="The notes are: {notes} and the questions are: {questions}. Aggregate them into a single document."
)

In [7]:
parallel_chain = RunnableParallel(
    {
        "notes": promptGenerateNotes | model | parser,
        "questions": promptGenerateQuestions | model | parser
    }
)

In [8]:
merge_chain = promptAggregator | model | parser

In [9]:
chain = parallel_chain | merge_chain

In [10]:
text = """
Here’s a long block of text you can use for testing (LLM calls, chunking, vector stores, etc.). It’s structured but continuous so you can also test splitting and retrieval.

---

The history of software development is, in many ways, the history of abstraction. In the earliest days of computing, programmers worked directly with machine code, encoding operations as numeric instructions that were painstakingly organized, debugged, and executed on hardware that filled entire rooms. Each line represented a tightly coupled dance between human intention and the unforgiving constraints of the machine’s architecture. Over time, assembly languages provided a small but meaningful layer of human readability, allowing programmers to use mnemonic codes and labels instead of raw numerical opcodes and addresses. This shift did not merely improve ergonomics; it enabled more complex systems to be conceived, shared, and maintained.

As high-level languages emerged—FORTRAN, COBOL, LISP, and later C—software development began to resemble the kind of structured thinking that scientists, business analysts, and mathematicians were already familiar with. Instead of thinking in terms of registers and memory locations, they could express loops, conditions, and data transformations in a syntax closer to human language. This evolution accelerated the pace at which software could be written and expanded the pool of people who could participate in its creation. As languages evolved, so did the tools surrounding them: compilers, linkers, debuggers, and later integrated development environments.

The arrival of object-oriented programming introduced another powerful layer of abstraction. Rather than treating data and behavior as separate concerns, objects allowed developers to bundle state and functionality together, making it easier to model real-world concepts or complex systems. Classes and inheritance hierarchies allowed for code reuse and polymorphism, and patterns such as encapsulation and abstraction provided guardrails for designing robust systems. Of course, these abstractions came with their own pitfalls: over-engineered class hierarchies, misplaced inheritance, and frameworks so complex that understanding them became a career specialty in itself. Yet, the core ideas of modularity and composition persisted and continue to influence modern paradigms, even as the industry moves toward functional programming, reactive systems, and microservices.

In parallel with these language-level changes, the way we deploy and operate software has undergone a revolution. Initially, deployment might have meant copying binaries onto a single server and updating configuration files by hand. Over time, configuration management tools, containerization, and orchestration frameworks such as Docker and Kubernetes transformed deployment from an artisanal craft into a reproducible, automated, and observable process. Infrastructure-as-code tools enabled developers and operators to treat infrastructure definitions with the same rigor as application code, checking them into version control and using CI/CD pipelines to apply changes in a controlled, testable manner.

One of the most profound recent shifts in software has been the rise of data-driven and machine learning–driven applications. Traditionally, software behavior was primarily dictated by explicit logic: if the input matches condition X, perform action Y. With machine learning, especially deep learning, behavior emerges from learned parameters optimized over large datasets. Instead of explicitly writing rules for every scenario, engineers define model architectures, training procedures, and evaluation metrics. This leads to systems that can handle tasks previously thought to be intractable for explicitly programmed logic, such as image recognition, natural language understanding, and game playing at superhuman levels. However, this also introduces new failure modes: opacity in decision-making, brittleness under distributional shift, and the need for robust data governance.

Generative models, particularly large language models, have opened yet another era in software. Instead of purely deterministic code driven by predefined rules, software systems can now incorporate components that generate text, images, code, or even synthetic data based on learned representations of the world. Large language models can draft documents, suggest code completions, summarize logs, and power conversational interfaces. This transition is not merely about automation; it represents a shift in how we think about human–computer interaction. Rather than requiring users to adapt to rigid interfaces and command structures, conversational systems can adapt to the user’s natural language, context, and intent.

A key challenge in this new era is orchestration: how to combine deterministic logic, data retrieval, and generative components into reliable systems. Frameworks like LangChain, semantic routers, and retrieval-augmented generation pipelines are attempts to impose structure on the inherently probabilistic nature of generative models. In a typical retrieval-augmented pipeline, a user query is first embedded into a high-dimensional vector space, a vector store is queried for relevant documents, and the retrieved context is then fed into a generative model that produces an answer grounded in external knowledge. This architecture bridges the gap between a model’s latent knowledge and the ever-changing reality reflected in external data stores, knowledge bases, and APIs.

Despite their potential, these systems are not magic. They are constrained by the quality and coverage of their training data, by the biases present in the sources they ingest, and by the engineering trade-offs made in model size, latency, and cost. For instance, a small local model running on a laptop will respond more quickly and privately than a gigantic cloud-hosted model, but may struggle with nuanced reasoning or very specialized knowledge. Conversely, a large frontier model may offer impressive performance but at the expense of higher latency, operational complexity, and the need to carefully manage access and cost. Engineers must therefore think in terms of model portfolios: selecting the right model for the right task, sometimes chaining multiple models together, and sometimes falling back to deterministic logic when reliability is paramount.

Observability is another crucial dimension. Traditional software engineering already emphasized logging, metrics, and traces to understand system behavior in production. With generative AI in the loop, observability must extend to prompts, responses, and the metadata that connects them: model versions, latency, token usage, and success metrics. Developers need tools to inspect not only whether the system is up, but also whether it is answering correctly, safely, and consistently over time. Evaluation becomes a continuous process, blending automated tests, offline benchmarks, and human feedback. New evaluation techniques—for instance, using models to evaluate other models—are emerging, along with frameworks that treat prompt templates and chains as first-class artifacts that can be versioned, tested, and rolled back.

Security and governance are equally vital. Introducing generative AI into workflows raises questions about data privacy, prompt injection, model exfiltration, and misuse of capabilities. Guardrails can be implemented at multiple levels: at the user interface, in the prompt construction, in intermediate validation steps, and in post-processing filters. Retrieval must be scoped carefully, ensuring that only authorized data is exposed to the model at query time. Policies about logging and retention must account for the fact that user inputs may contain personally identifiable information, proprietary secrets, or regulated content. Organizations must balance the desire for rich analytics with the obligation to protect individuals and comply with local and international regulations.

From the perspective of a developer learning these tools, the most important skill is not memorizing every function or configuration option, but understanding the conceptual building blocks from which systems are assembled. A modern AI-enabled application might combine: a frontend delivering a conversational interface or workflow UI; a backend orchestrating calls to LLMs, vector stores, and traditional databases; background jobs refreshing embeddings and indexes; and monitoring dashboards showing usage patterns and model performance over time. Each component can be understood with familiar engineering concepts: inputs, outputs, contracts, latency budgets, and failure modes. The novelty lies in the probabilistic nature of model outputs and the need to design systems that are resilient to uncertainty.

In practice, building robust systems with large language models means embracing iterative design. The first version of a prompt or chain is rarely optimal. Developers experiment with different structures, additional context, system messages that constrain behavior, and fallback flows for when the model’s answer is incomplete or incorrect. Over time, these experiments lead to patterns: reusable prompt templates, chain compositions, and integration strategies that handle common scenarios such as question answering, data extraction, code generation, and tool invocation. Community-driven ecosystems play a central role here, enabling practitioners to share examples, libraries, and best practices so that each team does not have to reinvent the wheel.

As organizations adopt these technologies at scale, the focus shifts from individual features to platform thinking. Instead of building one-off integrations, they design internal platforms that standardize how teams invoke models, store embeddings, manage secrets, and enforce governance policies. This platform layer becomes a strategic asset: it reduces duplication of effort, increases security, and fosters experimentation within a controlled environment. Teams can plug into this platform, confident that concerns such as authentication, rate limiting, logging, and basic safety checks are handled centrally. This, in turn, accelerates innovation, allowing domain experts to focus on their specific problems rather than on the plumbing required to wire up models and data.

The pace of change in this field is both exhilarating and daunting. Models grow larger and more capable; tooling becomes richer and more ergonomic; but the core challenges of good engineering remain remarkably stable. Clarity of requirements, correctness, maintainability, performance, security, and user experience never go out of fashion. Generative AI does not remove these concerns; it recontextualizes them. For example, correctness is no longer guaranteed by deterministic code paths alone, but by a combination of prompt design, retrieval quality, model evaluation, and guardrails. Performance is influenced not just by algorithmic complexity, but by token counts, network latency to model endpoints, and caching strategies for repeated queries.

Looking ahead, one can imagine development workflows where the boundary between human and machine contributions becomes increasingly fluid. A developer might describe a feature in natural language, and an assistant generates the initial scaffolding: API endpoints, database migrations, tests, and documentation. The developer reviews, refines, and corrects the output, iterating with the assistant until the feature meets quality standards. Over time, the assistant learns from these corrections, aligning itself more closely with the team’s conventions and architecture. Meanwhile, CI pipelines may incorporate AI-based code reviewers that highlight potential bugs, security issues, or design divergences, complementing traditional static analysis tools.

Crucially, this future does not eliminate the need for human judgment. If anything, it elevates it. Someone still needs to decide which problems are worth solving, which trade-offs are acceptable, and what constitutes “good enough” in a particular context. Human oversight remains essential in defining ethical boundaries, ensuring compliance, and interpreting ambiguous or conflicting requirements. The most effective practitioners will be those who can combine deep technical understanding with an appreciation of human factors: communication, collaboration, empathy, and a willingness to learn continuously as the tools evolve.

In summary, the evolution from machine code to high-level languages, from monoliths to microservices, and from deterministic rules to generative models reflects a broader pattern: we are continually seeking abstractions that allow us to build more powerful systems with less friction. Each new layer of abstraction opens opportunities while introducing fresh complexities. Generative AI is not the end of this journey but a new chapter, one in which software becomes more conversational, adaptive, and context-aware. For developers, data scientists, and organizations alike, the challenge is to harness these capabilities responsibly, thoughtfully, and creatively, turning potential into tangible value while remaining mindful of the risks and responsibilities that come with such transformative technology.

---

If you need an even larger text blob (e.g., multiple tens of thousands of characters for chunking tests), tell me roughly how many words or characters you want, and I’ll generate a longer one.

"""

In [11]:
chain.invoke({"data": text})

'# 🧠 High-Level Summary\n\nThe evolution of software development reflects a continuous trend toward **abstraction**, enabling more complex systems with greater ease and accessibility. This progression spans from early machine code through high-level languages, object-oriented programming, deployment automation, and now into data-driven and generative AI systems.\n\n---\n\n## 🔁 Evolution of Software Development\n\n### 1. **Early Days: Machine Code**\n- Programmers wrote directly in binary or assembly.\n- Extremely low-level, tightly coupled to hardware.\n- Required deep understanding of architecture; debugging was arduous.\n\n### 2. **Assembly Languages**\n- Introduced mnemonics and labels for readability.\n- Improved ergonomics and allowed more complex systems.\n\n### 3. **High-Level Languages (FORTRAN, COBOL, LISP, C)**\n- Enabled structured thinking familiar to scientists and analysts.\n- Allowed expression of loops, conditions, and transformations in human-readable syntax.\n- Accele